# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
!pip install pyspark requests

In [2]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [4]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

In [5]:
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [6]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

Add a column that creates a unique key to identify each record in order to answer questions about individual trips

In [7]:
from pyspark.sql.functions import monotonically_increasing_id

df_trips = df_trips.withColumn(
    "trip_id",
    monotonically_increasing_id()
)

In [8]:
df_trips.select("trip_id", "passenger_count", "trip_distance").show(5)

+-------+---------------+-------------+
|trip_id|passenger_count|trip_distance|
+-------+---------------+-------------+
|      0|            1.0|          1.5|
|      1|            1.0|          2.6|
|      2|            3.0|          0.0|
|      3|            5.0|          0.0|
|      4|            5.0|          0.0|
+-------+---------------+-------------+
only showing top 5 rows


Which trip has the highest passanger count


In [9]:
df_trips.orderBy("passenger_count", ascending=False) \
    .select("trip_id", "passenger_count") \
    .show(1)

+-------+---------------+
|trip_id|passenger_count|
+-------+---------------+
| 949956|            9.0|
+-------+---------------+
only showing top 1 row


What is the Average passanger count

In [10]:
from pyspark.sql.functions import avg

df_trips.select(
    avg("passenger_count").alias("average_passenger_count")
).show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



Shortest/longest trip by distance? by time?.

In [11]:
# Shortest trip by distance
df_trips.orderBy("trip_distance", ascending=True) \
    .select("trip_id", "trip_distance") \
    .show(1)

# Longest trip by distance
df_trips.orderBy("trip_distance", ascending=False) \
    .select("trip_id", "trip_distance") \
    .show(1)

+-------+-------------+
|trip_id|trip_distance|
+-------+-------------+
|      2|          0.0|
+-------+-------------+
only showing top 1 row


+-------+-------------+
|trip_id|trip_distance|
+-------+-------------+
|6074091|        831.8|
+-------+-------------+
only showing top 1 row


In [12]:
from pyspark.sql.functions import unix_timestamp

df_trips = df_trips.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp("tpep_dropoff_datetime") -
        unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

In [13]:
df_trips.select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes"
).show(5)

+-------+--------------------+---------------------+---------------------+
|trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+-------+--------------------+---------------------+---------------------+
|      0| 2019-01-01 00:46:40|  2019-01-01 00:53:20|    6.666666666666667|
|      1| 2019-01-01 00:59:47|  2019-01-01 01:18:59|                 19.2|
|      2| 2018-12-21 13:48:30|  2018-12-21 13:52:40|    4.166666666666667|
|      3| 2018-11-28 15:52:25|  2018-11-28 15:55:45|   3.3333333333333335|
|      4| 2018-11-28 15:56:57|  2018-11-28 15:58:33|                  1.6|
+-------+--------------------+---------------------+---------------------+
only showing top 5 rows


In [14]:
# Shortest trip by time
df_trips.orderBy("trip_duration_minutes") \
    .select("trip_id", "trip_duration_minutes") \
    .show(1)

# Longest trip by time
df_trips.orderBy("trip_duration_minutes", ascending=False) \
    .select("trip_id", "trip_duration_minutes") \
    .show(1)

+-------+---------------------+
|trip_id|trip_duration_minutes|
+-------+---------------------+
|1203184|             -84280.5|
+-------+---------------------+
only showing top 1 row


+-------+---------------------+
|trip_id|trip_duration_minutes|
+-------+---------------------+
|  68267|    43648.01666666667|
+-------+---------------------+
only showing top 1 row


busiest day/slowest single day

In [15]:
from pyspark.sql.functions import to_date, count

df_daily = df_trips.withColumn(
    "trip_date",
    to_date("tpep_pickup_datetime")
).groupBy("trip_date") \
 .agg(count("*").alias("number_of_trips"))

In [16]:
df_daily.orderBy("number_of_trips", ascending=False).show(1)

+----------+---------------+
| trip_date|number_of_trips|
+----------+---------------+
|2019-01-25|         292499|
+----------+---------------+
only showing top 1 row


In [17]:
df_daily.orderBy("number_of_trips", ascending=True).show(1)


+----------+---------------+
| trip_date|number_of_trips|
+----------+---------------+
|2019-05-20|              1|
+----------+---------------+
only showing top 1 row


busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )

In [18]:
from pyspark.sql.functions import hour, count

df_hourly = df_trips.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
).groupBy("pickup_hour") \
 .agg(count("*").alias("number_of_trips"))

In [19]:
# Busiest
df_hourly.orderBy("number_of_trips", ascending=False).show(1)

+-----------+---------------+
|pickup_hour|number_of_trips|
+-----------+---------------+
|         18|         515390|
+-----------+---------------+
only showing top 1 row


In [20]:
# Slowest
df_hourly.orderBy("number_of_trips", ascending=True).show(1)

+-----------+---------------+
|pickup_hour|number_of_trips|
+-----------+---------------+
|          4|          61424|
+-----------+---------------+
only showing top 1 row


On average which day of the week is slowest/busiest

In [21]:
from pyspark.sql.functions import date_format, avg, round

df_weekday_avg = df_daily.withColumn(
    "day_of_week",
    date_format("trip_date", "EEEE")
).groupBy("day_of_week") \
 .agg(
    round(avg("number_of_trips"), 2).alias("average_number_of_trips")
)
 # Busiest day of the week on average
df_weekday_avg.orderBy(
    "average_number_of_trips",
    ascending=False
).show(1)
# Slowest day of the week on average
df_weekday_avg.orderBy(
    "average_number_of_trips",
    ascending=True
).show(1)

+-----------+-----------------------+
|day_of_week|average_number_of_trips|
+-----------+-----------------------+
|   Thursday|              193863.29|
+-----------+-----------------------+
only showing top 1 row


+-----------+-----------------------+
|day_of_week|average_number_of_trips|
+-----------+-----------------------+
|     Monday|                90812.1|
+-----------+-----------------------+
only showing top 1 row


Does trip distance or num passangers affect tip amount

In [22]:
from pyspark.sql.functions import corr, round

df_trips.select(
    round(corr("trip_distance", "tip_amount"), 3)
        .alias("distance_tip_correlation"),

    round(corr("passenger_count", "tip_amount"), 3)
        .alias("passenger_tip_correlation")
).show()

+------------------------+-------------------------+
|distance_tip_correlation|passenger_tip_correlation|
+------------------------+-------------------------+
|                   0.527|                    0.001|
+------------------------+-------------------------+



The correlation between trip distance and tip amount is 0.527, so there is a moderate positive link. In general, longer trips tend to have higher tips.

The correlation between passenger count and tip amount is 0.001, so there is basically no link between the number of passengers and the tip amount.

What was the highest "extra" charge and which trip

In [23]:
df_trips.orderBy("extra", ascending=False) \
    .select("trip_id", "extra") \
    .show(1)

+-------+------+
|trip_id| extra|
+-------+------+
|5323483|535.38|
+-------+------+
only showing top 1 row


Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [24]:
# Check some extreme values
df_trips.select(
    "trip_id",
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "extra",
    "tip_amount"
).orderBy("extra", ascending=False).show(10)

df_trips.orderBy("trip_distance", ascending=False) \
    .select("trip_id", "trip_distance") \
    .show(10)

df_trips.orderBy("fare_amount", ascending=False) \
    .select("trip_id", "fare_amount") \
    .show(10)

+-------+---------------+-------------+---------------------+-----------+------+----------+
|trip_id|passenger_count|trip_distance|trip_duration_minutes|fare_amount| extra|tip_amount|
+-------+---------------+-------------+---------------------+-----------+------+----------+
|5323483|            1.0|          0.0|                  0.0|  355676.98|535.38|       0.0|
|7453230|            1.0|          0.0|                  0.0|        4.5| 23.04|       0.0|
| 543203|            1.0|         16.6|    72.88333333333334|       61.0|  18.5|       0.0|
| 311052|            1.0|        17.23|   43.666666666666664|       52.0|  18.5|       0.0|
| 548308|            1.0|        16.74|                36.35|       47.5|  18.5|     15.76|
|2455086|            1.0|        15.78|    49.38333333333333|       49.0|  18.5|     16.06|
|4017135|            1.0|        34.39|    65.63333333333334|       91.5|  18.5|       0.0|
| 134549|            1.0|         13.4|   38.416666666666664|       39.5|  18.5|

+-------+-------------+
|trip_id|trip_distance|
+-------+-------------+
|6074091|        831.8|
|4286633|        700.7|
|6770985|       214.01|
|4707534|       211.36|
|4881785|       201.27|
|4813335|       160.52|
|2567443|        144.2|
|4876419|       143.63|
|1144939|       142.88|
|4911314|        132.8|
+-------+-------------+
only showing top 10 rows


+-------+-----------+
|trip_id|fare_amount|
+-------+-----------+
|2499655|  623259.86|
|5323483|  355676.98|
|2159971|    36090.3|
|1892781|   34674.65|
|1649451|   33023.53|
| 651633|   31107.91|
|2444777|   30444.52|
| 737331|   30130.71|
|1419685|   25628.96|
| 301112|   25356.38|
+-------+-----------+
only showing top 10 rows


Outliers / strange values

- Extra charge of 535.38
- Trip distance of 831.8 miles
- Trip distance of 700.7 miles
- Fare amount of 623259.86
- Fare amount of 355676.98
- Trip 5323019: 0 mile and 0 min but fare amount = 355676.98

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [25]:
import requests

zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

response = requests.get(zone_url)

with open("taxi_zone_lookup.csv", "wb") as f:
    f.write(response.content)

zones = spark.read.csv(
    "taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

In [26]:
zones.show(10)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows


which borough had most pickups? dropoffs?



In [27]:
from pyspark.sql.functions import count

pickup_boroughs = df_trips.join(
    zones,
    df_trips.PULocationID == zones.LocationID,
    "left"
)

pickup_boroughs.groupBy("Borough") \
    .agg(count("*").alias("number_of_pickups")) \
    .orderBy("number_of_pickups", ascending=False) \
    .show(1)

+---------+-----------------+
|  Borough|number_of_pickups|
+---------+-----------------+
|Manhattan|          6950965|
+---------+-----------------+
only showing top 1 row


In [28]:
dropoff_boroughs = df_trips.join(
    zones,
    df_trips.DOLocationID == zones.LocationID,
    "left"
)

dropoff_boroughs.groupBy("Borough") \
    .agg(count("*").alias("number_of_dropoffs")) \
    .orderBy("number_of_dropoffs", ascending=False) \
    .show(1)

+---------+------------------+
|  Borough|number_of_dropoffs|
+---------+------------------+
|Manhattan|           6817355|
+---------+------------------+
only showing top 1 row


what are the busy/slow times by borough


In [29]:
from pyspark.sql.functions import hour, count
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

borough_hour = pickup_boroughs.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
).groupBy("Borough", "pickup_hour") \
 .agg(count("*").alias("number_of_trips"))

In [30]:
# Busiest hour by borough
window_busy = Window.partitionBy("Borough").orderBy(col("number_of_trips").desc())

busiest_by_borough = borough_hour.withColumn(
    "rank",
    row_number().over(window_busy)
).filter(col("rank") == 1)

busiest_by_borough.select(
    "Borough",
    "pickup_hour",
    "number_of_trips"
).show()

+-------------+-----------+---------------+
|      Borough|pickup_hour|number_of_trips|
+-------------+-----------+---------------+
|        Bronx|          7|           1803|
|     Brooklyn|          8|           6935|
|          EWR|         15|             54|
|    Manhattan|         18|         471539|
|          N/A|         19|            214|
|       Queens|         16|          29885|
|Staten Island|          8|             36|
|      Unknown|         18|          10751|
+-------------+-----------+---------------+



In [31]:
# Slowest hour by borough
window_slow = Window.partitionBy("Borough").orderBy(col("number_of_trips").asc())

slowest_by_borough = borough_hour.withColumn(
    "rank",
    row_number().over(window_slow)
).filter(col("rank") == 1)

slowest_by_borough.select(
    "Borough",
    "pickup_hour",
    "number_of_trips"
).show()

+-------------+-----------+---------------+
|      Borough|pickup_hour|number_of_trips|
+-------------+-----------+---------------+
|        Bronx|          3|            225|
|     Brooklyn|          3|           1919|
|          EWR|         23|              1|
|    Manhattan|          4|          53447|
|          N/A|          6|             88|
|       Queens|          3|           3085|
|Staten Island|          1|              3|
|      Unknown|          4|           1465|
+-------------+-----------+---------------+



what are the busiest days of the week by borough?


Borough refers to the pickup location. The busiest weekday is the one with the most pickups over the month.

In [32]:
from pyspark.sql.functions import date_format, count, col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

borough_day = pickup_boroughs.withColumn(
    "day_of_week",
    date_format("tpep_pickup_datetime", "EEEE")
).groupBy("Borough", "day_of_week") \
 .agg(count("*").alias("number_of_trips"))

window_day = Window.partitionBy("Borough").orderBy(
    col("number_of_trips").desc(), "day_of_week"
)

borough_day.withColumn("rank", row_number().over(window_day)) \
    .filter(col("rank") == 1) \
    .select("Borough", "day_of_week", "number_of_trips") \
    .show()

+-------------+-----------+---------------+
|      Borough|day_of_week|number_of_trips|
+-------------+-----------+---------------+
|        Bronx|   Thursday|           3121|
|     Brooklyn|    Tuesday|          15779|
|          EWR|  Wednesday|             83|
|    Manhattan|   Thursday|        1229554|
|          N/A|    Tuesday|            703|
|       Queens|   Thursday|          78972|
|Staten Island|     Friday|             64|
|      Unknown|   Thursday|          28929|
+-------------+-----------+---------------+



what is the average trip distance by borough?


In [33]:
from pyspark.sql.functions import avg, round

pickup_boroughs.groupBy("Borough") \
    .agg(round(avg("trip_distance"), 2).alias("average_trip_distance")) \
    .orderBy("average_trip_distance", ascending=False) \
    .show()

+-------------+---------------------+
|      Borough|average_trip_distance|
+-------------+---------------------+
|Staten Island|                 12.5|
|       Queens|                11.28|
|        Bronx|                 7.23|
|     Brooklyn|                 4.79|
|          N/A|                 3.19|
|          EWR|                 2.64|
|      Unknown|                 2.42|
|    Manhattan|                 2.23|
+-------------+---------------------+



what is the average trip fare by borough?


In [34]:
pickup_boroughs.groupBy("Borough") \
    .agg(round(avg("fare_amount"), 2).alias("average_trip_fare")) \
    .orderBy("average_trip_fare", ascending=False) \
    .show()

+-------------+-----------------+
|      Borough|average_trip_fare|
+-------------+-----------------+
|          EWR|            76.24|
|          N/A|            59.57|
|Staten Island|            45.29|
|       Queens|            35.14|
|        Bronx|            26.27|
|     Brooklyn|            18.65|
|      Unknown|            14.94|
|    Manhattan|            10.79|
+-------------+-----------------+



highest/lowest faire amounts for a trip, what burough is associated with the each

In [35]:
# Highest fare and its pickup borough
pickup_boroughs.orderBy(col("fare_amount").desc_nulls_last()) \
    .select("trip_id", "Borough", "fare_amount") \
    .show(1)

# Lowest fare and its pickup borough
pickup_boroughs.orderBy(col("fare_amount").asc_nulls_last()) \
    .select("trip_id", "Borough", "fare_amount") \
    .show(1)

+-------+---------+-----------+
|trip_id|  Borough|fare_amount|
+-------+---------+-----------+
|2499655|Manhattan|  623259.86|
+-------+---------+-----------+
only showing top 1 row


+-------+-------+-----------+
|trip_id|Borough|fare_amount|
+-------+-------+-----------+
|4890649| Queens|     -362.0|
+-------+-------+-----------+
only showing top 1 row


load the dataset from the most recently available january, is there a change to any of the average metrics.

The introduction asks for January 2025, while Part 2 asks for the latest January available. We compare both January 2025 and January 2026 with 2019 ([TLC data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)). The same averages are used for each year, without removing outliers.

In [36]:
from pyspark.sql.functions import lit

for year in [2019, 2025, 2026]:
    if year == 2019:
        df_year = df_trips
    else:
        download_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-01.parquet"
        response = requests.get(download_url)
        response.raise_for_status()

        filename = f"yellow_tripdata_{year}-01.parquet"
        with open(filename, "wb") as f:
            f.write(response.content)

        df_year = spark.read.parquet(filename)

    year_averages = df_year.select(
        lit(year).alias("year"),
        round(avg("passenger_count"), 2).alias("average_passenger_count"),
        round(avg("trip_distance"), 2).alias("average_trip_distance"),
        round(avg("fare_amount"), 2).alias("average_trip_fare")
    )

    if year == 2019:
        comparison = year_averages
    else:
        comparison = comparison.unionByName(year_averages)

comparison.orderBy("year").show()

+----+-----------------------+---------------------+-----------------+
|year|average_passenger_count|average_trip_distance|average_trip_fare|
+----+-----------------------+---------------------+-----------------+
|2019|                   1.57|                 2.83|            12.53|
|2025|                    1.3|                 5.86|            17.08|
|2026|                   1.26|                 6.46|             20.8|
+----+-----------------------+---------------------+-----------------+



In [37]:
# Changes compared with 2019 (distance in miles, fare in dollars)
results = comparison.orderBy("year").collect()
for result in results[1:]:
    print(f"January {result['year']} compared with January 2019:")
    for metric in ["average_passenger_count", "average_trip_distance", "average_trip_fare"]:
        change = result[metric] - results[0][metric]
        print(f"{metric}: {change:+.2f}")

January 2025 compared with January 2019:
average_passenger_count: -0.27
average_trip_distance: +3.03
average_trip_fare: +4.55
January 2026 compared with January 2019:
average_passenger_count: -0.31
average_trip_distance: +3.63
average_trip_fare: +8.27


### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [38]:
df_trips.createOrReplaceTempView("trips")
zones.createOrReplaceTempView("zones")

What is the average passenger count? (SQL)

In [39]:
spark.sql("""
    SELECT AVG(passenger_count) AS average_passenger_count
    FROM trips
""").show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



What was the highest extra charge and which trip? (SQL)

In [40]:
spark.sql("""
    SELECT trip_id, extra
    FROM trips
    ORDER BY extra DESC
    LIMIT 1
""").show()

+-------+------+
|trip_id| extra|
+-------+------+
|5323483|535.38|
+-------+------+



What is the average trip distance by borough? (SQL with a join)

In [41]:
spark.sql("""
    SELECT z.Borough,
           ROUND(AVG(t.trip_distance), 2) AS average_trip_distance
    FROM trips t
    LEFT JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY average_trip_distance DESC
""").show()

+-------------+---------------------+
|      Borough|average_trip_distance|
+-------------+---------------------+
|Staten Island|                 12.5|
|       Queens|                11.28|
|        Bronx|                 7.23|
|     Brooklyn|                 4.79|
|          N/A|                 3.19|
|          EWR|                 2.64|
|      Unknown|                 2.42|
|    Manhattan|                 2.23|
+-------------+---------------------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing